# Phase 1 — Dataset Curation & Exploration

This notebook builds the unified curated dataset by:
1. Linking Lakh MIDI files with MSD HDF5 metadata via `LakhMSDLinker`
2. Filtering by DTW match score (`min_score ≥ 0.70`)
3. Extracting harmonic features per MIDI using `MIDIHarmonicAnalyzer`
4. Saving the resulting DataFrame to Parquet for downstream phases

**Run this notebook from the `KG_DL_PROJECT/` root directory.**

In [ ]:
# ── Environment setup ──────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath('../..'))   # KG_DL_PROJECT root

import json
import pandas as pd
import numpy  as np
import matplotlib.pyplot as plt

from harmonic_kg_project.config.config   import CFG
from harmonic_kg_project.dataset         import MIDIHarmonicAnalyzer, LakhMSDLinker

print('Config loaded:')
print(f"  MIDI root : {CFG['data']['midi_root']}")
print(f"  H5 root   : {CFG['data']['h5_root']}")

In [ ]:
# ── Load match scores ──────────────────────────────────────────────────────
with open(CFG['data']['match_scores_path']) as f:
    match_scores = json.load(f)

all_scores  = np.array([s for trk in match_scores.values() for s in trk.values()])
best_scores = np.array([max(trk.values()) for trk in match_scores.values()])

print(f"  Total track IDs  : {len(match_scores):,}")
print(f"  Total MIDI files : {len(all_scores):,}")

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].hist(all_scores,  bins=80, alpha=0.7, label='all')
axes[0].hist(best_scores, bins=80, alpha=0.7, label='best/track')
axes[0].axvline(0.70, color='red', linestyle='--', label='thr=0.70')
axes[0].set_title('Match Score Distribution'); axes[0].legend()
axes[1].ecdf(best_scores); axes[1].axvline(0.70, color='red', linestyle='--')
axes[1].set_title('ECDF — best score per track')
plt.tight_layout(); plt.show()

In [ ]:
# ── Initialise analyzer + linker ───────────────────────────────────────────
analyzer = MIDIHarmonicAnalyzer(
    key_window    = CFG['analyzer']['key_window'],
    key_confidence= CFG['analyzer']['key_confidence'],
)
linker = LakhMSDLinker(
    midi_root    = CFG['data']['midi_root'],
    h5_root      = CFG['data']['h5_root'],
    analyzer     = analyzer,
    match_scores = match_scores,
)
print('✓ Analyzer and linker ready.')

In [ ]:
# ── Build dataset (prototype: 50 tracks) ───────────────────────────────────
# Change max_tracks=None to process the full dataset (takes hours)
MAX_TRACKS = 50

df = linker.build_dataset(
    min_score   = CFG['dataset']['min_match_score'],
    pick_midi   = CFG['dataset']['pick_midi'],
    max_tracks  = MAX_TRACKS,
    with_harmonic_features = True,
    verbose     = True,
)

print(f'\nDataFrame shape : {df.shape}')
df.head(3)

In [ ]:
# ── Explore: genre distribution ────────────────────────────────────────────
genre_counts = df['primary_genre'].value_counts()
top_n = min(20, len(genre_counts))

genre_counts[:top_n].plot(kind='barh', figsize=(9, 5), title=f'Top {top_n} genres')
plt.xlabel('# songs'); plt.tight_layout(); plt.show()
print(genre_counts[:top_n])

In [ ]:
# ── Explore: key & mode distribution ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
df['global_key'].value_counts()[:12].plot(kind='bar', ax=axes[0], title='Detected key')
df['global_mode'].value_counts().plot(kind='pie', ax=axes[1], title='Mode', autopct='%1.0f%%')
plt.tight_layout(); plt.show()

In [ ]:
# ── Explore: harmonic feature distributions ────────────────────────────────
harm_cols = ['transition_entropy', 'chord_vocab_roman', 'num_modulations',
             'harm_rhythm_mean', 'func_ratio_T', 'func_ratio_D']
avail = [c for c in harm_cols if c in df.columns]

df[avail].hist(bins=20, figsize=(12, 6), layout=(2, 3))
plt.suptitle('Harmonic Feature Distributions', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# ── Save dataset ───────────────────────────────────────────────────────────
import pathlib
out_dir = pathlib.Path(CFG['dataset']['output_dir'])
out_dir.mkdir(parents=True, exist_ok=True)

parquet_path = CFG['dataset']['parquet_file']
csv_path     = CFG['dataset']['csv_file']

df.to_parquet(parquet_path, index=False)
df.to_csv(csv_path, index=False)
print(f'✓ Saved to {parquet_path}')
print(f'✓ Saved to {csv_path}')